In [1]:
import os, glob, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# inspect a npz file
def inspect_npz(file_path):
    data = np.load(file_path)
    print(f"Contents of {file_path}:")
    for key in data.files:
        print(f" - {key}: shape {data[key].shape}, dtype {data[key].dtype}")
    return data
inspect_npz(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_2.5x_norm2\unspecified\test\FA 56B.npz")

Contents of C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_2.5x_norm2\unspecified\test\FA 56B.npz:
 - qq: shape (359, 16), dtype float32
 - coords: shape (359, 2), dtype int32
 - mask: shape (359,), dtype bool


NpzFile 'C:\\Users\\Vivian\\Documents\\PANTHER\\PANTHER\\patch_probs\\qq_2.5x_norm2\\unspecified\\test\\FA 56B.npz' with keys: qq, coords, mask

: 

In [2]:
def load_npz(npz_path: Path):
    d = np.load(npz_path, allow_pickle=False)
    qq = d["qq"]          # (N, K)
    coords = d["coords"]  # (N, 2)
    mask = d["mask"]      # (N,)
    return qq, coords, mask

def stats_for_slide(npz_path: Path, renorm=False, nearhard=0.99, atol=1e-4):
    slide_id = npz_path.stem
    qq, coords, mask = load_npz(npz_path)

    assert qq.ndim == 2, f"{slide_id}: qq should be (N,K)"
    N, K = qq.shape
    assert coords.shape[0] == N and mask.shape[0] == N, f"{slide_id}: shape mismatch"

    qq_m   = qq[mask]
    coords_m = coords[mask]
    Nm     = qq_m.shape[0]

    # row-sum check (before renorm)
    row_sums   = qq_m.sum(axis=1)
    row_sum_ok = np.allclose(row_sums, 1.0, atol=atol)
    row_min, row_max = float(row_sums.min()), float(row_sums.max())

    # optional renorm for clean stats
    if renorm:
        qq_m = qq_m / (row_sums[:, None] + 1e-12)
        row_sums = qq_m.sum(axis=1)
        row_sum_ok = np.allclose(row_sums, 1.0, atol=1e-6)
        row_min, row_max = float(row_sums.min()), float(row_sums.max())

    # entropy (natural log)
    eps = 1e-12
    ent = -(qq_m * np.log(qq_m + eps)).sum(axis=1)
    ent_mean, ent_std = float(ent.mean()), float(ent.std())

    # near-hard patches
    mx = qq_m.max(axis=1)
    nh_count = int((mx > nearhard).sum())
    nh_frac  = float(nh_count / max(1, Nm))

    # usage (argmax frequency) & average probability for ALL K
    top      = qq_m.argmax(axis=1)
    counts   = np.bincount(top, minlength=K)
    usage    = counts / counts.sum()
    avgprob  = qq_m.mean(axis=0)

    rec = {
        "slide_id": slide_id,
        "N_patches": int(Nm),
        "K_protos": int(K),
        "row_sum_ok": bool(row_sum_ok),
        "row_sum_min": row_min,
        "row_sum_max": row_max,
        "entropy_mean": ent_mean,
        "entropy_std": ent_std,
        "nearhard_thresh": float(nearhard),
        "nearhard_count": nh_count,
        "nearhard_frac": nh_frac,
    }
    for k in range(K):
        rec[f"usage_p{k}"]   = float(usage[k])
        rec[f"avgprob_p{k}"] = float(avgprob[k])

    # top proto details (5 strongest patches)
    p_star = int(usage.argmax())
    idx = np.argsort(-qq_m[:, p_star])[:5]
    rec["top_proto"]    = p_star
    rec["top5_probs"]   = ";".join(f"{qq_m[i, p_star]:.4f}" for i in idx)
    rec["top5_coords"]  = ";".join(f"{coords_m[i,0]},{coords_m[i,1]}" for i in idx)
    return rec


In [4]:
# Point this to the root folder that contains your split subfolders with .npz files
NPZ_ROOT = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_2.5x_norm2")

# Options
RENORM_ROWS   = True       # set True to renormalize each row of qq before stats
NEARHARD_THR  = 0.99       # threshold for "near-hard" assignment
CSV_OUT       = NPZ_ROOT / "qq_npz_summary3.csv"  # set to None to skip writing


In [5]:
npz_files = [Path(p) for p in glob.glob(str(NPZ_ROOT / "**" / "*.npz"), recursive=True)]
print(f"Found {len(npz_files)} npz files under {NPZ_ROOT}")

records = []
for p in npz_files:
    try:
        rec = stats_for_slide(p, renorm=RENORM_ROWS, nearhard=NEARHARD_THR)
        records.append(rec)
    except Exception as e:
        print(f"[ERROR] {p}: {e}")

df = pd.DataFrame.from_records(records).sort_values("slide_id").reset_index(drop=True)
display(df.head(10))  # preview

if CSV_OUT is not None:
    df.to_csv(CSV_OUT, index=False)
    print(f"Wrote CSV -> {CSV_OUT}")


Found 240 npz files under C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_2.5x_norm2


,slide_id,N_patches,K_protos,row_sum_ok,row_sum_min,row_sum_max,entropy_mean,entropy_std,nearhard_thresh,nearhard_count,...,avgprob_p12,usage_p13,avgprob_p13,usage_p14,avgprob_p14,usage_p15,avgprob_p15,top_proto,top5_probs,top5_coords
0,FA 100 B1,805,16,True,1.0,1.0,2.295871,0.205349,0.99,0,...,0.047349,0.083230,0.082169,0.029814,0.027144,0.038509,0.075048,1,0.4852;0.4546;0.4460;0.4347;0.4118,"3584,4480;3360,4480;6944,4480;2688,2464;2912,4480"
1,FA 100 B2,832,16,True,1.0,1.0,2.286666,0.238365,0.99,0,...,0.049228,0.036058,0.077408,0.022837,0.024189,0.082933,0.083090,1,0.4963;0.4857;0.4843;0.4705;0.4535,"7168,5152;7392,5152;4256,5600;7616,3360;4704,896"
2,FA 101 B1,588,16,True,1.0,1.0,2.248296,0.256342,0.99,0,...,0.036093,0.103741,0.099132,0.037415,0.033737,0.013605,0.062514,1,0.4700;0.4686;0.4652;0.4621;0.4132,"6496,2912;4256,2912;5824,3136;4032,2688;5600,2464"
3,FA 101 B2,659,16,True,1.0,1.0,2.215103,0.295910,0.99,0,...,0.040372,0.004552,0.050292,0.053111,0.048388,0.220030,0.116037,15,0.3520;0.3330;0.3182;0.3136;0.3112,"1792,4928;2016,4704;5376,5376;2016,4928;6048,5152"
4,FA 102 B1,731,16,True,1.0,1.0,2.160166,0.253333,0.99,0,...,0.017223,0.017784,0.053240,0.008208,0.020381,0.168263,0.089426,8,0.6264;0.5749;0.5631;0.5615;0.5599,"1792,4928;1344,4704;1120,4928;3808,6944;1344,4928"
5,FA 102 B2,399,16,True,1.0,1.0,2.233867,0.214466,0.99,0,...,0.041333,0.005013,0.053338,0.000000,0.021949,0.150376,0.088672,4,0.5034;0.4917;0.4774;0.4741;0.4702,"2464,4480;2016,4256;2464,2688;2240,672;5824,3360"
6,FA 103 B1,754,16,True,1.0,1.0,2.315016,0.192702,0.99,0,...,0.035761,0.074271,0.082373,0.007958,0.021064,0.051724,0.074076,7,0.4517;0.4122;0.3973;0.3946;0.3912,"4480,4928;4704,5376;5600,6048;4256,5152;4480,5376"
7,FA 103 B2,368,16,True,1.0,1.0,2.310062,0.213950,0.99,0,...,0.030013,0.105978,0.083983,0.013587,0.022570,0.029891,0.067955,7,0.4269;0.3977;0.3948;0.3934;0.3910,"3360,2016;3584,4032;3360,2240;2912,3808;2688,3360"
8,FA 104 B1,639,16,True,1.0,1.0,2.264918,0.243250,0.99,0,...,0.024226,0.029734,0.068532,0.029734,0.026591,0.015649,0.064674,9,0.3344;0.3126;0.2787;0.2705;0.2696,"7392,1344;7392,1568;6944,1120;7168,1568;6048,2016"
9,FA 104 B2,851,16,True,1.0,1.0,2.229004,0.292126,0.99,0,...,0.030210,0.010576,0.058216,0.019976,0.023871,0.029377,0.067548,9,0.2965;0.2905;0.2900;0.2895;0.2811,"7168,5376;7392,1344;7616,1792;7616,4256;6272,1120"


Wrote CSV -> C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_2.5x_norm2\qq_npz_summary3.csv


In [ ]:
def collect_matrix(df, prefix="usage_p"):
    cols = [c for c in df.columns if c.startswith(prefix)]
    cols = sorted(cols, key=lambda s: int(s.split(prefix)[1]))
    M    = df[cols].to_numpy()
    labels = [int(c.split(prefix)[1]) for c in cols]
    return M, labels

U, protos = collect_matrix(df, prefix="usage_p")

plt.figure(figsize=(max(8, U.shape[1]*0.5), max(4, U.shape[0]*0.3)))
im = plt.imshow(U, aspect="auto", interpolation="nearest")
plt.colorbar(im, fraction=0.046, pad=0.04, label="usage (argmax freq)")
plt.yticks(range(len(df)), df["slide_id"].astype(str).tolist(), fontsize=8)
plt.xticks(range(len(protos)), [f"P{k}" for k in protos], fontsize=8)
plt.title("Prototype usage per slide")
plt.tight_layout()
plt.show()


In [ ]:
MAX_SLIDES = 30  # change as you like
S, K = U.shape
idx = np.arange(min(S, MAX_SLIDES))
cum = np.zeros(len(idx))

plt.figure(figsize=(max(10, len(idx)*0.4), 6))
for k in range(K):
    plt.bar(idx, U[:len(idx), k], bottom=cum, label=f"P{k}")
    cum += U[:len(idx), k]

plt.xticks(idx, df.loc[idx, "slide_id"].astype(str).tolist(), rotation=90, fontsize=8)
plt.ylabel("usage (argmax freq)")
plt.title(f"Stacked prototype usage (first {len(idx)} slides)")
plt.legend(ncol=min(4, K), fontsize=8)
plt.tight_layout()
plt.show()


plots only


In [ ]:
# ---- plots-only from CSV ----
import pandas as pd, numpy as np, matplotlib.pyplot as plt

CSV_PATH = r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_5x_norm2\qq_npz_summary2.csv"   # <-- set me
df = pd.read_csv(CSV_PATH)

def collect_matrix(df, prefix="usage_p"):
    cols = sorted([c for c in df.columns if c.startswith(prefix)],
                  key=lambda s: int(s.split(prefix)[1]))
    M = df[cols].to_numpy()
    labels = [int(c.split(prefix)[1]) for c in cols]
    return M, labels

# Choose "usage_p" (argmax freq) or "avgprob_p" (avg prob mass)
U, protos = collect_matrix(df, prefix="usage_p")

# Heatmap
plt.figure(figsize=(max(8, U.shape[1]*0.5), max(4, U.shape[0]*0.3)))
im = plt.imshow(U, aspect="auto", interpolation="nearest")
plt.colorbar(im, fraction=0.046, pad=0.04, label="usage (argmax freq)")
plt.yticks(range(len(df)), df["slide_id"].astype(str).tolist(), fontsize=8)
plt.xticks(range(len(protos)), [f"P{k}" for k in protos], fontsize=8)
plt.title("Prototype usage per slide")
plt.tight_layout()
plt.show()

# Stacked bars (first N slides)
MAX_SLIDES = 30
S, K = U.shape
idx = np.arange(min(S, MAX_SLIDES))
cum = np.zeros(len(idx))
plt.figure(figsize=(max(10, len(idx)*0.4), 6))
for k in range(K):
    plt.bar(idx, U[:len(idx), k], bottom=cum, label=f"P{k}")
    cum += U[:len(idx), k]
plt.xticks(idx, df.loc[idx, "slide_id"].astype(str).tolist(), rotation=90, fontsize=8)
plt.ylabel("usage (argmax freq)")
plt.title(f"Stacked prototype usage (first {len(idx)} slides)")
plt.legend(ncol=min(4, K), fontsize=8)
plt.tight_layout()
plt.show()
